# SHL 2026 — Grammar Scoring v6 (reliability-first: honest evaluation + data fixes)
Same feature pipeline as v5 (every cache reused — a cheap rerun), but the modelling section is
rebuilt around two **verified** facts about the data (see §8b): 37 training labels are an
anomalous `0` batch outside the 1–5 rubric, and the train set is 60s-dominant while the test set
is 45s-dominant with duration correlated to score. v6 therefore:

1. **Fixes the target** — compares training on all-769 vs 732-positive, scored on the same 1–5 targets.
2. **Fixes the validation** — outer-CV of the *second stage* (NNLS blend + isotonic fit inside each
   outer-train only); base-model OOF is per-row honest (single-pass). Compare configs by the delta.
3. **Reports a test-like (duration-reweighted) RMSE** beside random-CV — the number that predicts the LB.
4. **Tests, not assumes** — a compact grammar specialist and a −CTC ablation are kept only if they
   improve the honest, test-like held-out score. Deploys the winner; clips to [1,5].


## 0 · CONFIG

In [ ]:
# ---- data / CV (same backbone as v4) ----
SEEDS       = [42, 1, 2025]
N_FOLDS     = 5
SSL_PCA_DIM = 64

# ---- Whisper ASR (unchanged from v4; cache reused) ----
ASR_MODEL      = 'deepdml/faster-whisper-large-v3-turbo-ct2'  # fallback -> 'large-v3' -> 'small'
ASR_BEAM       = 5

# ---- v4 signals (kept ON) ----
USE_DEJONG   = True      # timing/fluency from word timestamps
USE_ASR_CONF = True      # decoder-confidence features (v4's #1 signal, 23.4%)
USE_GEC      = True      # grammatical-error-correction edit statistics
GEC_MODEL    = 'pszemraj/flan-t5-large-grammar-synthesis'
GEC_PREFIX   = ''
GEC_MAX_SENTS= 14
GEC_BATCH    = 16
USE_LLM_JUDGE = True     # zero-shot 0-5 grammar rating -> feature + optional blend member
LLM_MODEL     = 'Qwen/Qwen2.5-3B-Instruct'   # proven to load on T4; 7B is a better judge if memory allows
LLM_AS_BLEND  = True
USE_SSL       = True

# ---- v5: verbatim-transcript grammar recovery (the fix for Whisper's normalisation) ----
USE_CTC        = True    # verbatim CTC transcript; if it fails to load, v5 self-disables and runs as v4
CTC_MODEL      = 'facebook/wav2vec2-large-960h'   # exact model already used for SSL -> guaranteed available
                                                  # (fallbacks: -large-960h-lv60-self, -base-960h)
USE_DIVERGENCE = True    # Whisper<->CTC edit distance features (dv_*)  -- cheapest, most robust new signal
USE_CTC_LING   = True    # verbatim error/disfluency features (c_*)
GEC_ON_CTC     = True    # feed GEC the verbatim transcript instead of Whisper's laundered text
LLM_ON_CTC     = True    # feed the LLM judge the verbatim transcript

# ---- modelling upgrades ----
ADD_HUBER      = True    # extra LGB-Huber blend member (robust to noisy 769-row MOS labels)
APPLY_CALIBRATION = True # bagged/single isotonic, chosen on honest nested delta
print('CONFIG loaded (v5)')
# ---- v6: reliability (honest evaluation + data fixes) ----
HANDLE_ZEROS   = True   # compare all-769 vs 732-positive; deploy the pre-registered DEPLOY_CONFIG
DUR_SHORT_S    = 50.0   # split short(45s)/long(60s) duration cohorts
OUTER_FOLDS    = 5
OUTER_SEEDS    = [0,1,2]
ABLATE_CTC     = True   # also report the -CTC held-out ablation
DEPLOY_CONFIG  = 'B pos-732'  # pre-registered deploy; avoids winner's-curse min-selection. None -> pure min
SELECT_MARGIN_SE = 1.0   # only override the default if a config beats it by >this many fold-SE
print('v6 reliability config loaded')

## 1 · Setup

In [ ]:
import subprocess, sys, importlib
def pip(pkgs):
    for p in pkgs:
        ok=False
        for extra in (['--only-binary=:all:'], []):
            try:
                subprocess.run([sys.executable,'-m','pip','install','-q',*extra,p],check=True,timeout=600)
                print('ok  :',p); ok=True; break
            except Exception as e: last=e
        if not ok: print('SKIP:',p,'->',type(last).__name__)
pip(['lightgbm','faster-whisper','sentencepiece','accelerate','language-tool-python'])
import os, gc, json, warnings, math, re, random, difflib
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, librosa, torch
from pathlib import Path
from tqdm.auto import tqdm
SEED=SEEDS[0]; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
HAS_PRAAT=importlib.util.find_spec('parselmouth') is not None
HAS_LT   =importlib.util.find_spec('language_tool_python') is not None
def cache_pq(name, build):
    if Path(name).exists(): print('cache hit :',name); return pd.read_parquet(name)
    df=build(); df.to_parquet(name); print('cached    :',name); return df
print('device:',DEVICE,' praat:',HAS_PRAAT,' langtool:',HAS_LT)

### 1b · Cache bootstrap — makes this a CHEAP rerun (reuse v5 features, no re-extraction)

In [ ]:
# A NEW Kaggle notebook starts with an EMPTY /kaggle/working, so nothing is cached by default and
# every heavy cell would recompute (the full ~3h run). To make v6 cheap: Add Input -> Notebook Output
# -> your v5 run, and this cell copies its cached artifacts into the working dir so each ASR/SSL/GEC/
# LLM cell hits cache (minutes, not hours). If nothing is found, the pipeline still runs correctly --
# it just recomputes. (Cheapest of all: instead re-run §9+ inside the SAME v5 notebook, which already
# holds the caches.)
import glob, shutil, os
CACHE_GLOBS=['ac_*.parquet','w2v_*.parquet','wl_*.parquet','lg_*_v4.parquet','asrmeta_*_v4.json',
             'ctc_*_v5.json','ctcfeat_*_v5.parquet','gec_*.parquet','llm_*.parquet']
copied=0
for base in glob.glob('/kaggle/input/*'):
    for pat in CACHE_GLOBS:
        for srcf in glob.glob(os.path.join(base,'**',pat),recursive=True):
            dst=os.path.basename(srcf)
            if not os.path.exists(dst):
                try: shutil.copy(srcf,dst); copied+=1
                except Exception as e: print('copy fail',srcf,e)
present=sorted(os.path.basename(p) for p in glob.glob('*.parquet')+glob.glob('*_v*.json'))
print(f'cache bootstrap: copied {copied} artifact(s); {len(present)} cache file(s) now present')
if present: print('  e.g.:', present[:12])
else:       print('  no caches found -> this will be a FULL (slow) run; attach the v5 output to make it cheap')

## 2 · Load data (keyed to test.csv — never sample_submission)

In [ ]:
CANDIDATES=["/kaggle/input/shl-hiring-assessment-2026","/kaggle/input/shl-hiring-assessment-2026/Dataset_Final"]
DATA=next((Path(p) for p in CANDIDATES if Path(p).exists()),None)
if DATA is None:
    import kagglehub; DATA=Path(kagglehub.competition_download('shl-hiring-assessment-2026'))
def find(rgx):
    for p in DATA.rglob('*'):
        if re.search(rgx,p.name,re.I): return p
train_csv=find(r'^train\.csv$'); test_csv=find(r'^test\.csv$')
train_wav_dir=next((p for p in DATA.rglob('*') if p.is_dir() and 'audios_train' in p.name.lower()),None) \
           or next((p for p in DATA.rglob('*') if p.is_dir() and p.name.lower() in ('train','train_audios','audio_train')),None)
test_wav_dir =next((p for p in DATA.rglob('*') if p.is_dir() and 'audios_test' in p.name.lower()),None) \
           or next((p for p in DATA.rglob('*') if p.is_dir() and p.name.lower() in ('test','test_audios','audio_test')),None)
train=pd.read_csv(train_csv); test=pd.read_csv(test_csv)
LABEL_COL='label' if 'label' in train.columns else [c for c in train.columns if c!='filename'][0]
FILE_COL ='filename' if 'filename' in train.columns else train.columns[0]
print(train.shape,test.shape,'|',train_wav_dir,test_wav_dir)
print('label range:',float(train[LABEL_COL].min()),'..',float(train[LABEL_COL].max()),
      '| mean',round(float(train[LABEL_COL].mean()),3),'| std',round(float(train[LABEL_COL].std()),3))

## 3 · Acoustic features (cached — reused from v3/v4, ASR-independent)

In [ ]:
if HAS_PRAAT:
    import parselmouth
    from parselmouth.praat import call
SR=16000
def acoustic_feats(path):
    y,sr=librosa.load(path,sr=SR,mono=True); dur=len(y)/sr
    mfcc=librosa.feature.mfcc(y=y,sr=sr,n_mfcc=20)
    d1=librosa.feature.delta(mfcc); d2=librosa.feature.delta(mfcc,order=2)
    feats={}
    for name,arr in [('mfcc',mfcc),('d1',d1),('d2',d2)]:
        feats[f'{name}_mean']=arr.mean(); feats[f'{name}_std']=arr.std()
        for i in range(arr.shape[0]): feats[f'{name}{i}_m']=arr[i].mean(); feats[f'{name}{i}_s']=arr[i].std()
    for n,a in [('sc',librosa.feature.spectral_centroid(y=y,sr=sr)[0]),
                ('sbw',librosa.feature.spectral_bandwidth(y=y,sr=sr)[0]),
                ('sro',librosa.feature.spectral_rolloff(y=y,sr=sr)[0]),
                ('zcr',librosa.feature.zero_crossing_rate(y)[0]),
                ('rms',librosa.feature.rms(y=y)[0])]:
        feats[f'{n}_m']=a.mean(); feats[f'{n}_s']=a.std()
    iv=librosa.effects.split(y,top_db=30); speech=sum((b-a) for a,b in iv)/sr
    feats.update(dur=dur,speech_ratio=speech/dur,n_pauses=len(iv)-1,
                 mean_pause=np.mean([iv[i+1,0]-iv[i,1] for i in range(len(iv)-1)])/sr if len(iv)>1 else 0)
    try:
        assert HAS_PRAAT
        snd=parselmouth.Sound(y,sampling_frequency=sr); pv=snd.to_pitch().selected_array['frequency']; pv=pv[pv>0]
        feats['f0_m']=pv.mean() if len(pv) else 0; feats['f0_s']=pv.std() if len(pv) else 0
        pp=call(snd,"To PointProcess (periodic, cc)",75,500)
        feats['jitter']=call(pp,"Get jitter (local)",0,0,1e-4,0.02,1.3)
        feats['shimmer']=call([snd,pp],"Get shimmer (local)",0,0,1e-4,0.02,1.3,1.6)
    except Exception: feats.update(f0_m=0,f0_s=0,jitter=0,shimmer=0)
    return feats
def build_ac(df,wav_dir,tag):
    def _b():
        rows=[acoustic_feats(str(wav_dir/f)) for f in tqdm(df[FILE_COL],desc=f'ac-{tag}')]
        out=pd.DataFrame(rows); out.insert(0,FILE_COL,df[FILE_COL].values); return out
    return cache_pq(f'ac_{tag}.parquet',_b)
ac_tr=build_ac(train,train_wav_dir,'tr'); ac_te=build_ac(test,test_wav_dir,'te')
print(ac_tr.shape)

## 4 · Whisper ASR — faster-whisper large-v3-turbo (cached _v4) → transcripts + timing + confidence
Unchanged from v4 (cache reused). One decode pass yields the transcript (feeds §5/§6/§7), word
timestamps (De-Jong fluency), and decoder confidence (v4's strongest signal). NOTE: this transcript
is *normalised* — that is exactly why v5 also computes a verbatim CTC transcript in §4c.


In [ ]:
def build_asr_v4():
    from faster_whisper import WhisperModel
    def _load(name):
        return WhisperModel(name, device='cuda' if DEVICE=='cuda' else 'cpu',
                            compute_type='float16' if DEVICE=='cuda' else 'int8')
    model=None
    for cand in [ASR_MODEL,'large-v3','small']:
        try: model=_load(cand); print('ASR model loaded:',cand); break
        except Exception as e: print('ASR load FAILED',cand,'->',repr(e))
    if model is None:
        raise RuntimeError('faster-whisper could not load ANY model (turbo/large-v3/small). '
                           'Check GPU + Internet=ON on Kaggle; do not proceed with empty transcripts.')
    def _one(path):
        # decode with librosa and pass the ndarray -> bypasses faster-whisper's PyAV loader,
        # which clashes with Kaggle's `av` ('metadata_errors' TypeError).  [v4 bug fix, preserved]
        y,_=librosa.load(str(path),sr=16000,mono=True); y=y.astype(np.float32)
        segments,info=model.transcribe(y,language='en',beam_size=ASR_BEAM,
                                       word_timestamps=True,vad_filter=True)
        segs=[]; words=[]
        for s in segments:
            segs.append(dict(text=s.text,alp=float(s.avg_logprob),nsp=float(s.no_speech_prob),
                             cr=float(s.compression_ratio),start=float(s.start),end=float(s.end)))
            for w in (s.words or []):
                words.append([float(w.start),float(w.end),float(w.probability)])
        return dict(dur=float(info.duration),text=' '.join(x['text'].strip() for x in segs).strip(),
                    segs=segs,words=words)
    def _run(df,wav_dir,tag):
        meta={}; empt=0
        for f in tqdm(df[FILE_COL],desc=f'asr-{tag}'):
            try:
                r=_one(wav_dir/f); meta[f]=r
                if not r['text']: empt+=1
            except Exception as e: print('asr fail',f,repr(e)); meta[f]=dict(dur=0.0,text='',segs=[],words=[]); empt+=1
        print(f'asr-{tag}: {empt}/{len(df)} empty/failed transcripts')
        return meta
    tr=_run(train,train_wav_dir,'tr'); te=_run(test,test_wav_dir,'te')
    del model; gc.collect(); torch.cuda.empty_cache(); return tr,te

def _meanwords(meta,df): return float(np.mean([len((meta[f]['text'] or '').split()) for f in df[FILE_COL]]))
if Path('asrmeta_tr_v4.json').exists() and Path('asrmeta_te_v4.json').exists():
    meta_tr=json.load(open('asrmeta_tr_v4.json')); meta_te=json.load(open('asrmeta_te_v4.json'))
    if _meanwords(meta_tr,train)==0:
        print('!! cached ASR meta is EMPTY -> deleting and recomputing'); os.remove('asrmeta_tr_v4.json'); os.remove('asrmeta_te_v4.json')
        meta_tr,meta_te=build_asr_v4()
    else: print('cache hit : asr meta v4')
else:
    meta_tr,meta_te=build_asr_v4()
tx_tr=pd.DataFrame({FILE_COL:train[FILE_COL].values,'text':[meta_tr[f]['text'] for f in train[FILE_COL]]})
tx_te=pd.DataFrame({FILE_COL:test[FILE_COL].values ,'text':[meta_te[f]['text'] for f in test[FILE_COL]]})
tx_tr['text']=tx_tr['text'].fillna(''); tx_te['text']=tx_te['text'].fillna('')
mw=float(tx_tr['text'].str.split().apply(len).mean())
print('mean Whisper words (train):',round(mw,1),'| sample:',repr(tx_tr['text'].iloc[0][:120]))
assert mw>0, ('ASR produced EMPTY transcripts -> GEC/LLM/fluency would be dead. Fix faster-whisper '
              '(see ASR load lines above) and re-run. Nothing cached.')
json.dump(meta_tr,open('asrmeta_tr_v4.json','w')); json.dump(meta_te,open('asrmeta_te_v4.json','w'))

### 4b · De-Jong fluency + ASR-confidence features (from the cached Whisper meta)

In [ ]:
PAUSE_MIN=0.25; PAUSE_LONG=0.60
def dejong_feats(m):
    dur=m.get('dur',0.0); ws=sorted(m.get('words',[]),key=lambda x:x[0])
    if dur<=0 or len(ws)==0:
        return dict(dj_nwords=0,dj_speech_rate=0,dj_artic_rate=0,dj_pause_ratio=0,dj_npauses=0,
                    dj_pause_permin=0,dj_longpause_permin=0,dj_mean_run=0,dj_std_run=0,dj_max_run=0,dj_mean_pause=0)
    gaps=[max(0.0,ws[i][0]-ws[i-1][1]) for i in range(1,len(ws))]
    paus=[g for g in gaps if g>=PAUSE_MIN]; longp=[g for g in gaps if g>=PAUSE_LONG]
    sil=float(sum(paus)); speech=max(1e-6,dur-sil); nw=len(ws); mins=max(1e-6,dur/60.0)
    runs=[]; cur=1
    for g in gaps:
        if g>=PAUSE_MIN: runs.append(cur); cur=1
        else: cur+=1
    runs.append(cur)
    return dict(dj_nwords=nw,dj_speech_rate=nw/dur,dj_artic_rate=nw/speech,dj_pause_ratio=sil/dur,
                dj_npauses=len(paus),dj_pause_permin=len(paus)/mins,dj_longpause_permin=len(longp)/mins,
                dj_mean_run=float(np.mean(runs)),dj_std_run=float(np.std(runs)),dj_max_run=float(np.max(runs)),
                dj_mean_pause=float(np.mean(paus)) if paus else 0.0)
def asrconf_feats(m):
    segs=m.get('segs',[]); ws=m.get('words',[])
    if not segs:
        return dict(ac_alp_mean=-2.0,ac_alp_min=-2.0,ac_alp_std=0,ac_nsp_mean=1.0,ac_nsp_max=1.0,
                    ac_cr_mean=0,ac_cr_max=0,ac_wconf_mean=0,ac_wconf_min=0,ac_wconf_std=0,ac_lowconf_rate=1.0)
    alp=[s['alp'] for s in segs]; nsp=[s['nsp'] for s in segs]; cr=[s['cr'] for s in segs]
    pr=[w[2] for w in ws] or [0.0]
    return dict(ac_alp_mean=np.mean(alp),ac_alp_min=np.min(alp),ac_alp_std=np.std(alp),
                ac_nsp_mean=np.mean(nsp),ac_nsp_max=np.max(nsp),ac_cr_mean=np.mean(cr),ac_cr_max=np.max(cr),
                ac_wconf_mean=np.mean(pr),ac_wconf_min=np.min(pr),ac_wconf_std=np.std(pr),
                ac_lowconf_rate=float(np.mean([1.0 if p<0.5 else 0.0 for p in pr])))
def build_extra(df,meta,tag):
    rows=[]
    for f in df[FILE_COL]:
        m=meta[f]; d={}
        if USE_DEJONG:   d.update(dejong_feats(m))
        if USE_ASR_CONF: d.update(asrconf_feats(m))
        rows.append(d)
    out=pd.DataFrame(rows) if rows and (USE_DEJONG or USE_ASR_CONF) else pd.DataFrame(index=range(len(df)))
    out.insert(0,FILE_COL,df[FILE_COL].values); return out
ex_tr=build_extra(train,meta_tr,'tr'); ex_te=build_extra(test,meta_te,'te')
print('dejong+asrconf dims:',ex_tr.shape[1]-1)

## 4c · Verbatim CTC transcript (wav2vec2-large-960h) — the fix for Whisper's grammar-normalisation
Whisper writes *fluent* text even from ungrammatical speech (v4 confirmed this killed GEC/LLM).
A CTC acoustic model has **no language-model decoder**, so it transcribes verbatim — false starts,
agreement errors, broken syntax survive. We greedily decode it (cheap: one forward pass, no beam,
no timestamps) and reuse `facebook/wav2vec2-large-960h` — the exact model already downloaded for SSL
in §8, so availability is guaranteed. This block is an **enhancement**: if it can't load or returns
empty, it disables itself and v5 runs identically to v4.


In [ ]:
def build_ctc_v5():
    from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor
    proc=mdl=None
    for cand in [CTC_MODEL,'facebook/wav2vec2-large-960h','facebook/wav2vec2-base-960h']:
        try:
            proc=Wav2Vec2Processor.from_pretrained(cand)
            mdl=Wav2Vec2ForCTC.from_pretrained(cand).to(DEVICE).eval()
            print('CTC model loaded:',cand); break
        except Exception as e: print('CTC load FAILED',cand,'->',repr(e)); proc=mdl=None
    if mdl is None: return None,None
    @torch.no_grad()
    def _one(path):
        y,_=librosa.load(str(path),sr=16000,mono=True); y=y.astype(np.float32)
        step=20*16000; parts=[]
        for i in range(0,max(1,len(y)),step):
            c=y[i:i+step]
            if len(c)<4000: continue
            iv=proc(c,sampling_rate=16000,return_tensors='pt').input_values.to(DEVICE)
            ids=torch.argmax(mdl(iv).logits,dim=-1)
            parts.append(proc.batch_decode(ids)[0])
        return ' '.join(p.strip() for p in parts).strip().lower()
    def _run(df,wav_dir,tag):
        d={}
        for f in tqdm(df[FILE_COL],desc=f'ctc-{tag}'):
            try: d[f]=_one(wav_dir/f)
            except Exception as e: print('ctc fail',f,repr(e)); d[f]=''
        return d
    tr=_run(train,train_wav_dir,'tr'); te=_run(test,test_wav_dir,'te')
    del mdl,proc; gc.collect(); torch.cuda.empty_cache(); return tr,te

def _ctc_meanwords(m,df): return float(np.mean([len((m.get(f,'') or '').split()) for f in df[FILE_COL]]))
ctc_tr=ctc_te=None
if USE_CTC:
    if Path('ctc_tr_v5.json').exists() and Path('ctc_te_v5.json').exists():
        ctc_tr=json.load(open('ctc_tr_v5.json')); ctc_te=json.load(open('ctc_te_v5.json'))
        if _ctc_meanwords(ctc_tr,train)==0:
            print('!! cached CTC is EMPTY -> recomputing'); ctc_tr=ctc_te=None
        else: print('cache hit : ctc v5')
    if ctc_tr is None:
        ctc_tr,ctc_te=build_ctc_v5()
    if ctc_tr is not None:
        mwc=_ctc_meanwords(ctc_tr,train)
        print('mean CTC words (train):',round(mwc,1),'| sample:',repr((ctc_tr.get(train[FILE_COL].iloc[0],''))[:140]))
        if mwc>0:
            json.dump(ctc_tr,open('ctc_tr_v5.json','w')); json.dump(ctc_te,open('ctc_te_v5.json','w'))
        else:
            print('!! CTC produced EMPTY transcripts -> disabling CTC levers this run'); ctc_tr=ctc_te=None
if ctc_tr is None:
    print('>> CTC disabled -> v5 runs with v4 signals only (safe fallback)'); USE_CTC=False
# side-by-side sanity peek: Whisper (laundered) vs CTC (verbatim)
if USE_CTC:
    f0=train[FILE_COL].iloc[0]
    print('  WHISPER:',repr((tx_tr['text'].iloc[0])[:140]))
    print('  CTC    :',repr((ctc_tr.get(f0,''))[:140]))

## 5 · Linguistic features on the WHISPER transcript (recomputed on turbo transcripts, cached _v4)

In [ ]:
import nltk; [nltk.download(x,quiet=True) for x in ('punkt','averaged_perceptron_tagger','punkt_tab','averaged_perceptron_tagger_eng')]
lt=None
if HAS_LT:
    import language_tool_python
    try: lt=language_tool_python.LanguageTool('en-US')
    except Exception as e: print('LT init failed:',e); lt=None
from transformers import GPT2LMHeadModel, GPT2TokenizerFast
from collections import Counter
FILLERS={'um','uh','erm','er','hmm','mm','mmm','umm','uhh','ah','eh'}
def _mattr(w,win=50):
    if len(w)<=win: return len(set(w))/max(1,len(w))
    return float(np.mean([len(set(w[i:i+win]))/win for i in range(len(w)-win+1)]))
def build_ling(tx,tag):
    def _b():
        gpt_tok=GPT2TokenizerFast.from_pretrained('gpt2'); gpt=GPT2LMHeadModel.from_pretrained('gpt2').to(DEVICE).eval()
        @torch.no_grad()
        def ppl(text,stride=512,max_len=1024):
            if not text.strip(): return 200.0
            enc=gpt_tok(text,return_tensors='pt').input_ids.to(DEVICE); nll,n=0.0,0
            for i in range(0,enc.size(1),stride):
                ch=enc[:,i:i+max_len]
                if ch.size(1)<2: break
                nll+=gpt(ch,labels=ch).loss.item()*(ch.size(1)-1); n+=ch.size(1)-1
            return math.exp(nll/max(n,1))
        def feats(text):
            text=text or ""; low=text.lower(); words=re.findall(r"\w+",low)
            sents=nltk.sent_tokenize(text) if text else []
            n_w,n_s=max(1,len(words)),max(1,len(sents))
            errs=lt.check(text) if (text and lt is not None) else []
            pc=Counter(t for _,t in (nltk.pos_tag(words) if words else []))
            slens=[len(re.findall(r"\w+",s.lower())) for s in sents] or [n_w]
            reps=sum(1 for i in range(1,len(words)) if words[i]==words[i-1])
            return {'n_words':n_w,'n_sents':n_s,'avg_wlen':np.mean([len(w) for w in words]) if words else 0,
                    'avg_slen':n_w/n_s,'ttr':len(set(words))/n_w,'err_rate':len(errs)/n_w,'n_errs':len(errs),
                    'ppl':ppl(text[:4000]),'log_ppl':math.log1p(ppl(text[:4000])),
                    **{f'pos_{k}':pc.get(k,0)/n_w for k in ['NN','VB','JJ','RB','PRP','DT','IN','CC','MD']},
                    'filler_rate':sum(low.count(f' {f} ') for f in FILLERS)/n_w,
                    'immed_repeat_rate':reps/n_w,'mattr50':_mattr(words),
                    'longword_rate':sum(1 for w in words if len(w)>=7)/n_w,'comma_rate':text.count(',')/n_w,
                    'slen_std':float(np.std(slens)),'subordinate_rate':pc.get('IN',0)/n_w,
                    'content_ratio':sum(pc.get(k,0) for k in ['NN','VB','JJ','RB'])/n_w}
        rows=[feats(t) for t in tqdm(tx['text'],desc=f'ling-{tag}')]
        del gpt,gpt_tok; gc.collect(); torch.cuda.empty_cache()
        out=pd.DataFrame(rows); out.insert(0,FILE_COL,tx[FILE_COL].values); return out
    return cache_pq(f'lg_{tag}_v4.parquet',_b)
lg_tr=build_ling(tx_tr,'tr'); lg_te=build_ling(tx_te,'te')
print(lg_tr.shape)

### 5b · Whisper↔CTC divergence + verbatim CTC error/disfluency features (cached _v5)
`dv_*` measure how much Whisper cleaned up the verbatim CTC text — high divergence (low `dv_sim`,
high `dv_edit_rate`) ⇒ the speaker was more disfluent/ungrammatical, so we expect a **negative**
correlation with the grammar label. `c_*` read grammatical-error and disfluency signals off the
**verbatim** transcript, where they survive. Every feature is neutral-valued when a transcript is
missing, so this block can never crash. Empty (all zero) when CTC is disabled.

In [ ]:
def _divergence(wh, ct):
    a=(wh or '').lower().split(); b=(ct or '').split()
    if len(a)==0 and len(b)==0:
        return dict(dv_sim=1.0,dv_edit_rate=0.0,dv_ins_rate=0.0,dv_del_rate=0.0,dv_sub_rate=0.0,
                    dv_jaccard=1.0,dv_len_ratio=1.0)
    sm=difflib.SequenceMatcher(a=a,b=b); ins=dele=sub=0
    for op,i1,i2,j1,j2 in sm.get_opcodes():
        if op=='replace': sub+=max(i2-i1,j2-j1)
        elif op=='insert': ins+=j2-j1
        elif op=='delete': dele+=i2-i1
    na=max(1,len(a)); sA,sB=set(a),set(b)
    return dict(dv_sim=sm.ratio(),dv_edit_rate=(ins+dele+sub)/na,dv_ins_rate=ins/na,dv_del_rate=dele/na,
                dv_sub_rate=sub/na,dv_jaccard=len(sA&sB)/max(1,len(sA|sB)),dv_len_ratio=len(b)/na)
def build_ctcfeat(tx, ctc_map, tag):
    def _b():
        gpt_tok=GPT2TokenizerFast.from_pretrained('gpt2'); gpt=GPT2LMHeadModel.from_pretrained('gpt2').to(DEVICE).eval()
        @torch.no_grad()
        def ppl(text,max_len=1024):
            t=(text or '').strip()
            if not t: return 200.0
            enc=gpt_tok(t,return_tensors='pt').input_ids.to(DEVICE)
            if enc.size(1)<2: return 200.0
            ch=enc[:,:max_len]; return math.exp(gpt(ch,labels=ch).loss.item())
        def feats(wh, ct):
            d=_divergence(wh,ct)
            w=(ct or '').split(); n=max(1,len(w))
            reps=sum(1 for i in range(1,len(w)) if w[i]==w[i-1])
            bg=sum(1 for i in range(3,len(w)) if w[i]==w[i-2] and w[i-1]==w[i-3])
            errs=lt.check(ct) if (ct and lt is not None) else []
            p=ppl((ct or '')[:4000])
            d.update(c_nwords=len(w),c_ttr=len(set(w))/n,c_mattr=_mattr(w),
                     c_repeat_rate=reps/n,c_bigram_repeat=bg/n,
                     c_filler_rate=sum(1 for x in w if x in FILLERS)/n,
                     c_err_rate=(len(errs)/n) if lt is not None else 0.0,
                     c_nerrs=float(len(errs)) if lt is not None else 0.0,
                     c_ppl=p,c_log_ppl=math.log1p(p))
            return d
        pairs=list(zip(tx[FILE_COL],tx['text']))
        rows=[feats(wh, ctc_map.get(f,'')) for f,wh in tqdm(pairs,desc=f'ctcfeat-{tag}')]
        del gpt,gpt_tok; gc.collect(); torch.cuda.empty_cache()
        out=pd.DataFrame(rows); out.insert(0,FILE_COL,tx[FILE_COL].values); return out
    return cache_pq(f'ctcfeat_{tag}_v5.parquet',_b)
if USE_CTC and (USE_DIVERGENCE or USE_CTC_LING):
    cf_tr=build_ctcfeat(tx_tr,ctc_tr,'tr'); cf_te=build_ctcfeat(tx_te,ctc_te,'te')
    print('ctc-feature dims:',cf_tr.shape[1]-1)
    _q=train.merge(cf_tr,on=FILE_COL)
    for c in ['dv_edit_rate','dv_sim','c_err_rate','c_repeat_rate']:
        if c in _q.columns:
            try: print(f'  corr({c:14s}, label) = {np.corrcoef(_q[c],_q[LABEL_COL].astype(float))[0,1]:+.3f}')
            except Exception: pass
else:
    cf_tr=pd.DataFrame({FILE_COL:train[FILE_COL].values}); cf_te=pd.DataFrame({FILE_COL:test[FILE_COL].values})

## 6 · GEC error features (cached) — now fed the VERBATIM transcript
A grammar-error-correction model rewrites the text; we measure how much it had to change (edit rate
+ ins/del/sub + fraction changed + worst-window rate). v4 fed it Whisper's laundered text ⇒ 0.9% of
gain. v5 feeds it the **verbatim CTC transcript** (`GEC_ON_CTC`), where the errors survive. Since CTC
text has no sentence punctuation, we split into fixed ~20-word windows as pseudo-sentences.

In [ ]:
if USE_CTC and GEC_ON_CTC:
    gsrc_tr=pd.DataFrame({FILE_COL:train[FILE_COL].values,'text':[ctc_tr.get(f,'') for f in train[FILE_COL]]})
    gsrc_te=pd.DataFrame({FILE_COL:test[FILE_COL].values ,'text':[ctc_te.get(f,'') for f in test[FILE_COL]]})
    GEC_SFX='c_v5'; print('GEC input: VERBATIM CTC transcript')
else:
    gsrc_tr,gsrc_te=tx_tr,tx_te; GEC_SFX='v4'; print('GEC input: Whisper transcript (v4 fallback)')
def _gec_chunks(text):
    sents=[s.strip() for s in (nltk.sent_tokenize(text) if text else []) if s.strip()]
    out=[]
    for s in sents:
        w=s.split()
        if len(w)<=22: out.append(s)
        else:
            for i in range(0,len(w),20): out.append(' '.join(w[i:i+20]))
    return out[:GEC_MAX_SENTS]
def build_gec(tx,tag):
    def _b():
        from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
        tok=AutoTokenizer.from_pretrained(GEC_MODEL)
        mdl=AutoModelForSeq2SeqLM.from_pretrained(GEC_MODEL).to(DEVICE).eval()
        @torch.no_grad()
        def correct(sents):
            xs=[GEC_PREFIX+s for s in sents]
            inp=tok(xs,return_tensors='pt',padding=True,truncation=True,max_length=128).to(DEVICE)
            out=mdl.generate(**inp,max_length=128,num_beams=4)
            return tok.batch_decode(out,skip_special_tokens=True)
        def diff(o,c):
            a=o.split(); b=c.split(); sm=difflib.SequenceMatcher(a=a,b=b); ins=dele=sub=0
            for op,i1,i2,j1,j2 in sm.get_opcodes():
                if op=='replace': sub+=max(i2-i1,j2-j1)
                elif op=='insert': ins+=j2-j1
                elif op=='delete': dele+=i2-i1
            return ins,dele,sub,max(1,len(a))
        def feats(text):
            sents=_gec_chunks(text)
            if not sents:
                return dict(gec_edit_rate=0,gec_mean_rate=0,gec_max_rate=0,gec_frac_changed=0,
                            gec_ins_rate=0,gec_del_rate=0,gec_sub_rate=0,gec_nsents=0)
            corr=[]
            for i in range(0,len(sents),GEC_BATCH): corr+=correct(sents[i:i+GEC_BATCH])
            te=tn=ti=td=ts=0; per=[]; chg=0
            for o,c in zip(sents,corr):
                i_,d_,s_,n=diff(o,c); e=i_+d_+s_; te+=e; tn+=n; ti+=i_; td+=d_; ts+=s_
                per.append(e/n); chg+=int(e>0)
            return dict(gec_edit_rate=te/max(1,tn),gec_mean_rate=float(np.mean(per)),
                        gec_max_rate=float(np.max(per)),gec_frac_changed=chg/len(sents),
                        gec_ins_rate=ti/max(1,tn),gec_del_rate=td/max(1,tn),gec_sub_rate=ts/max(1,tn),
                        gec_nsents=len(sents))
        rows=[feats(t) for t in tqdm(tx['text'],desc=f'gec-{tag}')]
        del mdl,tok; gc.collect(); torch.cuda.empty_cache()
        out=pd.DataFrame(rows); out.insert(0,FILE_COL,tx[FILE_COL].values); return out
    return cache_pq(f'gec_{tag}_{GEC_SFX}.parquet',_b)
if USE_GEC:
    gec_tr=build_gec(gsrc_tr,'tr'); gec_te=build_gec(gsrc_te,'te'); print('GEC dims:',gec_tr.shape[1]-1)
    _q=train.merge(gec_tr,on=FILE_COL)
    try: print(f'  corr(gec_edit_rate, label) = {np.corrcoef(_q["gec_edit_rate"],_q[LABEL_COL].astype(float))[0,1]:+.3f}')
    except Exception: pass
else:
    gec_tr=pd.DataFrame({FILE_COL:train[FILE_COL].values}); gec_te=pd.DataFrame({FILE_COL:test[FILE_COL].values})

## 7 · Zero-shot LLM grammar judge (cached) — now fed the VERBATIM transcript
An instruction model rates grammar 0–5 against the rubric. v4 fed it Whisper's clean text ⇒ NNLS
weight 0. v5 feeds it the **verbatim CTC transcript** (`LLM_ON_CTC`) so it can see the actual errors.
Used as a feature (trees exploit it) and, if it survives the finite-guard, an NNLS blend member. It
never sees labels ⇒ leakage-free. [v4 bug fix preserved: apply_chat_template return_dict + slice by
input_ids.shape[1].]

In [ ]:
RUBRIC=("You are a strict English grammar examiner scoring VERBATIM transcripts of spoken English "
        "(from a CTC recogniser, so they contain disfluencies, repetitions and no punctuation/casing "
        "-- judge the grammar of what was said, ignore transcription noise). Rate ONLY grammatical "
        "accuracy and sentence structure on a 0-5 scale: 5=high accuracy, well-controlled complex "
        "grammar; 4=good control, minor errors; 3=decent but clear/frequent errors; 2=basic structures "
        "with consistent mistakes, incomplete sentences; 1=struggles with basic sentence structure. "
        "Decimals allowed. Reply with ONLY the number.")
if USE_CTC and LLM_ON_CTC:
    lsrc_tr=pd.DataFrame({FILE_COL:train[FILE_COL].values,'text':[ctc_tr.get(f,'') for f in train[FILE_COL]]})
    lsrc_te=pd.DataFrame({FILE_COL:test[FILE_COL].values ,'text':[ctc_te.get(f,'') for f in test[FILE_COL]]})
    LLM_SFX='c_v5'; print('LLM input: VERBATIM CTC transcript')
else:
    lsrc_tr,lsrc_te=tx_tr,tx_te; LLM_SFX='v4'; print('LLM input: Whisper transcript (v4 fallback)')
def build_llm(tx,tag):
    def _b():
        from transformers import AutoTokenizer, AutoModelForCausalLM
        tok=AutoTokenizer.from_pretrained(LLM_MODEL)
        mdl=AutoModelForCausalLM.from_pretrained(LLM_MODEL,torch_dtype=torch.float16 if DEVICE=='cuda' else torch.float32,
                                                 device_map='auto').eval()
        dbg=[0]
        @torch.no_grad()
        def judge(text):
            text=(text or '').strip()
            if not text: return np.nan
            msg=[{'role':'system','content':RUBRIC},
                 {'role':'user','content':'Transcript:\n'+text[:2400]+'\n\nGrammar score (0-5):'}]
            enc=tok.apply_chat_template(msg,add_generation_prompt=True,return_tensors='pt',
                                        return_dict=True).to(mdl.device)
            out=mdl.generate(**enc,max_new_tokens=8,do_sample=False)
            dec=tok.decode(out[0][enc['input_ids'].shape[1]:],skip_special_tokens=True)
            if dbg[0]<3: print(f'  llm raw[{dbg[0]}]:',repr(dec)); dbg[0]+=1
            m=re.search(r'([0-5](?:\.\d+)?)',dec); return float(m.group(1)) if m else np.nan
        vals=[judge(t) for t in tqdm(tx['text'],desc=f'llm-{tag}')]
        del mdl,tok; gc.collect(); torch.cuda.empty_cache()
        out=pd.DataFrame({FILE_COL:tx[FILE_COL].values,'llm_score':vals}); return out
    if Path(f'llm_{tag}_{LLM_SFX}.parquet').exists(): print('cache hit : llm',tag); return pd.read_parquet(f'llm_{tag}_{LLM_SFX}.parquet')
    df=_b(); df.to_parquet(f'llm_{tag}_{LLM_SFX}.parquet'); return df
if USE_LLM_JUDGE:
    llm_tr=build_llm(lsrc_tr,'tr'); llm_te=build_llm(lsrc_te,'te')
    both=pd.concat([llm_tr['llm_score'],llm_te['llm_score']]); cov=int(both.notna().sum())
    ymean=float(train[LABEL_COL].astype(float).mean())
    fill=float(np.nanmean(both)) if cov>0 else ymean
    if not np.isfinite(fill): fill=ymean
    llm_tr['llm_score']=llm_tr['llm_score'].fillna(fill); llm_te['llm_score']=llm_te['llm_score'].fillna(fill)
    print(f'llm judge coverage: {cov}/{len(both)}  fill={fill:.3f}')
    if cov>0:
        _q=train.merge(llm_tr,on=FILE_COL)
        try: print(f'  corr(llm_score, label) = {np.corrcoef(_q["llm_score"],_q[LABEL_COL].astype(float))[0,1]:+.3f}')
        except Exception: pass
    if cov==0:
        print('!! LLM judge returned NO parseable scores (see raw[] above) -> disabling LLM lever this run')
        USE_LLM_JUDGE=False
        llm_tr=pd.DataFrame({FILE_COL:train[FILE_COL].values}); llm_te=pd.DataFrame({FILE_COL:test[FILE_COL].values})
else:
    llm_tr=pd.DataFrame({FILE_COL:train[FILE_COL].values}); llm_te=pd.DataFrame({FILE_COL:test[FILE_COL].values})

## 8 · SSL raw embeddings (cached — reused from v3/v4; mean-all-layers + mean/std pooling; PCA in-fold in §9)

In [ ]:
from transformers import AutoFeatureExtractor, AutoModel
MIN_SAMPLES=16000
def load_ssl(name):
    return AutoFeatureExtractor.from_pretrained(name), AutoModel.from_pretrained(name,output_hidden_states=True).to(DEVICE).eval()
@torch.no_grad()
def ssl_embed(path,fe,mdl,chunk_s=20):
    try:
        y,_=librosa.load(path,sr=16000,mono=True)
        if len(y)<MIN_SAMPLES: y=np.pad(y,(0,MIN_SAMPLES-len(y)))
        step=chunk_s*16000
        chunks=[c for c in (y[i:i+step] for i in range(0,len(y),step)) if len(c)>=MIN_SAMPLES] \
               or [np.pad(y,(0,max(0,MIN_SAMPLES-len(y))))]
        vecs=[]
        for c in chunks:
            inp=fe(c,sampling_rate=16000,return_tensors='pt').input_values.to(DEVICE)
            st=torch.stack(mdl(inp).hidden_states,0)
            m=st.mean(2).mean(0).squeeze(0); s=st.std(2).mean(0).squeeze(0)
            vecs.append(torch.cat([m,s]).float().cpu().numpy())
        return np.mean(vecs,0)
    except Exception as e:
        print('ssl fail',path,e); return None
def build_ssl(model_name, prefix):
    def _one(df,wav_dir,tag,fe,mdl):
        rows=[ssl_embed(str(wav_dir/f),fe,mdl) for f in tqdm(df[FILE_COL],desc=f'{prefix}-{tag}')]
        dim=max((len(r) for r in rows if r is not None),default=1)
        rows=[r if r is not None else np.zeros(dim,np.float32) for r in rows]
        E=np.vstack(rows).astype(np.float32)
        out=pd.DataFrame(E,columns=[f'{prefix}_raw{i}' for i in range(E.shape[1])])
        out.insert(0,FILE_COL,df[FILE_COL].values); return out
    if Path(f'{prefix}_tr.parquet').exists() and Path(f'{prefix}_te.parquet').exists():
        print('cache hit :',prefix); return pd.read_parquet(f'{prefix}_tr.parquet'), pd.read_parquet(f'{prefix}_te.parquet')
    fe,mdl=load_ssl(model_name)
    tr=_one(train,train_wav_dir,'tr',fe,mdl); te=_one(test,test_wav_dir,'te',fe,mdl)
    del mdl,fe; gc.collect(); torch.cuda.empty_cache()
    tr.to_parquet(f'{prefix}_tr.parquet'); te.to_parquet(f'{prefix}_te.parquet'); return tr,te
if USE_SSL:
    w2_tr,w2_te=build_ssl('facebook/wav2vec2-large-960h','w2v')
    wl_tr,wl_te=build_ssl('microsoft/wavlm-large','wl')
    print('w2v',w2_tr.shape,'wavlm',wl_tr.shape)
else:
    w2_tr=w2_te=wl_tr=wl_te=None

## 8b · Reliability diagnostics — label anomalies, train/test shift, speaker grouping

In [ ]:
yv=train[LABEL_COL].astype(float).values
print('== LABEL DISTRIBUTION =='); print(pd.Series(yv).value_counts().sort_index().to_string())
nz=int((yv==0).sum())
print(f'\nzero-labels: {nz}  (label 0 is OUTSIDE the 1-5 rubric -> handled in §9)')
print('predict-mean RMSE:  all-769=%.4f   positive-only=%.4f  (zeros inflate the metric)'%(
      np.sqrt(((yv-yv.mean())**2).mean()), np.sqrt(((yv[yv>0]-yv[yv>0].mean())**2).mean())))
if nz>0:
    zf=list(train[FILE_COL][yv==0]); print('zero-label files (sample):',zf[:2],'...',zf[-1:])
# train/test duration shift (cached acoustic 'dur')
if 'dur' in ac_tr.columns and 'dur' in ac_te.columns:
    dtr=train.merge(ac_tr[[FILE_COL,'dur']],on=FILE_COL)['dur'].values; dte=ac_te['dur'].values
    print('\n== DURATION SHIFT (drives test-like weighting in §9) ==')
    print('train-positive median=%.1fs | test median=%.1fs'%(np.median(dtr[yv>0]),np.median(dte)))
    print('short(<=%.0fs) share:  train=%.0f%%  test=%.0f%%'%(DUR_SHORT_S,
          100*np.mean(dtr[yv>0]<=DUR_SHORT_S),100*np.mean(dte<=DUR_SHORT_S)))
    for nm,m in [('train short',(dtr<=DUR_SHORT_S)&(yv>0)),('train long',(dtr>DUR_SHORT_S)&(yv>0))]:
        print('  %-11s n=%d  mean label=%.3f'%(nm,int(m.sum()),yv[m].mean()))
# speaker-grouping heuristic (wavlm cosine; captures content+speaker -> a HEURISTIC, not proof)
if USE_SSL and (wl_tr is not None):
    def _nrm(df):
        c=[x for x in df.columns if x.startswith('wl_raw')][:1024]
        X=df[c].values.astype('float32'); return X/(np.linalg.norm(X,axis=1,keepdims=True)+1e-8)
    E=_nrm(wl_tr); S=E@E.T; np.fill_diagonal(S,-1.0); nnc=S.max(1)
    print('\n== SPEAKER-GROUPING HEURISTIC (train nearest-neighbour cosine, wavlm) ==')
    print('  >0.99: %d   >0.97: %d   >0.95: %d   (of %d)'%((nnc>0.99).sum(),(nnc>0.97).sum(),(nnc>0.95).sum(),len(nnc)))
    print('  many high-similarity pairs => probable repeated speakers => prefer GroupKFold; few => random CV is fine.')

## 9 · HONEST evaluation — fix the target, fix the validation, then judge changes
Two verified data facts (§8b) drive this: (1) 37 zero-labels are an anomalous batch outside the 1-5
rubric; (2) train is 60s-dominant but test is 45s-dominant and duration correlates with score. So:
- compare **all-769** vs **732-positive** training, both scored on the SAME 1-5 targets;
- run an **outer-CV of the second stage** (NNLS weights + isotonic fit inside each outer-train only) so
  the blend/calibration isn't in-sample optimistic (base-model OOF is per-row honest, single-pass);
- report a **test-like (duration-reweighted) RMSE** beside random-CV RMSE (+ per-cohort, + Pearson) —
  the test-like number is the LB predictor;
- test a **grammar specialist** and a **-CTC** ablation, keeping each only if it helps the honest
  test-like score. Deploy the winner; clip to [1,5].

In [ ]:
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.linear_model import Ridge, HuberRegressor
from sklearn.svm import SVR
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.isotonic import IsotonicRegression
from scipy.stats import pearsonr
from scipy.optimize import nnls
from sklearn.metrics import mean_squared_error
from functools import reduce
def rmse(a,b): return float(np.sqrt(mean_squared_error(a,b)))
def wrmse(a,b,w):
    a,b,w=np.asarray(a,float),np.asarray(b,float),np.asarray(w,float)
    return float(np.sqrt(np.sum(w*(a-b)**2)/np.sum(w)))

# ---- assemble the full merged frame (identical blocks to v5) ----
blocks=[ac_tr,lg_tr,ex_tr]; blocks_te=[ac_te,lg_te,ex_te]
if USE_CTC and (USE_DIVERGENCE or USE_CTC_LING): blocks.append(cf_tr); blocks_te.append(cf_te)
if USE_GEC: blocks.append(gec_tr); blocks_te.append(gec_te)
if USE_LLM_JUDGE: blocks.append(llm_tr); blocks_te.append(llm_te)
if USE_SSL: blocks+=[w2_tr,wl_tr]; blocks_te+=[w2_te,wl_te]
Mtr=reduce(lambda a,b:a.merge(b,on=FILE_COL),blocks); Mte=reduce(lambda a,b:a.merge(b,on=FILE_COL),blocks_te)
y=train.merge(Mtr[[FILE_COL]],on=FILE_COL)[LABEL_COL].astype(float).values
w2c=[c for c in Mtr.columns if c.startswith('w2v_raw')]; wlc=[c for c in Mtr.columns if c.startswith('wl_raw')]
handc_all=[c for c in Mtr.columns if c!=FILE_COL and c not in w2c and c not in wlc]
def _v(df,cols): return np.nan_to_num(df[cols].values.astype(np.float32))
Wtr,Wte=(_v(Mtr,w2c),_v(Mte,w2c)) if USE_SSL else (np.zeros((len(y),0),'float32'),np.zeros((len(Mte),0),'float32'))
Ltr,Lte=(_v(Mtr,wlc),_v(Mte,wlc)) if USE_SSL else (np.zeros((len(y),0),'float32'),np.zeros((len(Mte),0),'float32'))
# duration per training row + test-cohort weights
dur_tr=(train.merge(ac_tr[[FILE_COL,'dur']],on=FILE_COL)['dur'].values if 'dur' in ac_tr.columns else np.full(len(y),50.0))
dur_te=(ac_te['dur'].values if 'dur' in ac_te.columns else np.full(len(Mte),50.0))
te_short_frac=float(np.mean(dur_te<=DUR_SHORT_S))
def testlike_w(dur_sub):
    short=dur_sub<=DUR_SHORT_S; f=float(short.mean())
    w=np.where(short, te_short_frac/max(1e-6,f), (1-te_short_frac)/max(1e-6,1-f)); return w/max(1e-9,w.mean())
bins=pd.cut(y,bins=[-0.1,1.5,2.5,3.5,4.5,5.1],labels=False).astype(int)
CTC_PREF=('dv_','c_'); GRAM_PREF=('dv_','c_','gec_')
GRAM_EXTRA=[c for c in ['subordinate_rate','avg_slen','slen_std','err_rate','n_errs','mattr50'] if c in handc_all]
if 'llm_score' in handc_all: GRAM_EXTRA=GRAM_EXTRA+['llm_score']
lgb_params=dict(objective='regression',metric='rmse',learning_rate=0.03,num_leaves=31,feature_fraction=0.7,
                bagging_fraction=0.8,bagging_freq=1,min_data_in_leaf=8,lambda_l2=1.0,verbose=-1)
lgbh_params={**lgb_params,'objective':'huber','alpha':0.9}
def infold_pca(Rtr,Rte,tr,dim,seed):
    if Rtr.shape[1]==0: return np.zeros((Rtr.shape[0],0)),np.zeros((Rte.shape[0],0))
    sc=StandardScaler().fit(Rtr[tr]); ztr=sc.transform(Rtr); zte=sc.transform(Rte)
    k=int(min(dim,ztr.shape[1],len(tr)-1)); p=PCA(n_components=k,random_state=seed).fit(ztr[tr])
    return p.transform(ztr),p.transform(zte)

def train_stack(train_idx, keep_cols, add_gram=False):
    '''Honest per-row OOF (filled only at train_idx positions) + fold-averaged test preds.'''
    A=_v(Mtr,keep_cols); Ae=_v(Mte,keep_cols)
    members=['lgb','lgbh','rdg','svr','et']+(['gram'] if add_gram else [])
    oof={m:np.zeros(len(y)) for m in members}; pred={m:np.zeros(len(Ae)) for m in members}
    if add_gram:
        gcols=[c for c in keep_cols if c.startswith(GRAM_PREF) or c in GRAM_EXTRA] or keep_cols[:8]
        G=_v(Mtr,gcols); Ge=_v(Mte,gcols)
    sub_bins=bins[train_idx]
    for seed in SEEDS:
        for trl,val in StratifiedKFold(N_FOLDS,shuffle=True,random_state=seed).split(train_idx,sub_bins):
            tr=train_idx[trl]; va=train_idx[val]
            w2p,w2pe=infold_pca(Wtr,Wte,tr,SSL_PCA_DIM,seed); wlp,wlpe=infold_pca(Ltr,Lte,tr,SSL_PCA_DIM,seed)
            Xtr=np.hstack([A,w2p,wlp]); Xte=np.hstack([Ae,w2pe,wlpe])
            m=lgb.train({**lgb_params,'seed':seed},lgb.Dataset(Xtr[tr],y[tr]),4000,
                        valid_sets=[lgb.Dataset(Xtr[va],y[va])],callbacks=[lgb.early_stopping(150),lgb.log_evaluation(0)])
            oof['lgb'][va]+=m.predict(Xtr[va])/len(SEEDS); pred['lgb']+=m.predict(Xte)/(N_FOLDS*len(SEEDS))
            mh=lgb.train({**lgbh_params,'seed':seed},lgb.Dataset(Xtr[tr],y[tr]),4000,
                         valid_sets=[lgb.Dataset(Xtr[va],y[va])],callbacks=[lgb.early_stopping(150),lgb.log_evaluation(0)])
            oof['lgbh'][va]+=mh.predict(Xtr[va])/len(SEEDS); pred['lgbh']+=mh.predict(Xte)/(N_FOLDS*len(SEEDS))
            et=ExtraTreesRegressor(n_estimators=600,max_features=0.5,min_samples_leaf=3,n_jobs=-1,random_state=seed).fit(Xtr[tr],y[tr])
            oof['et'][va]+=et.predict(Xtr[va])/len(SEEDS); pred['et']+=et.predict(Xte)/(N_FOLDS*len(SEEDS))
            sc=StandardScaler().fit(Xtr[tr]); Ztr,Zva,Zte=sc.transform(Xtr[tr]),sc.transform(Xtr[va]),sc.transform(Xte)
            r=Ridge(alpha=8.0).fit(Ztr,y[tr]); oof['rdg'][va]+=r.predict(Zva)/len(SEEDS); pred['rdg']+=r.predict(Zte)/(N_FOLDS*len(SEEDS))
            s=SVR(kernel='rbf',C=10,epsilon=0.2).fit(Ztr,y[tr]); oof['svr'][va]+=s.predict(Zva)/len(SEEDS); pred['svr']+=s.predict(Zte)/(N_FOLDS*len(SEEDS))
            if add_gram:
                scg=StandardScaler().fit(G[tr]); gtr,gva,gte=scg.transform(G[tr]),scg.transform(G[va]),scg.transform(Ge)
                try:
                    hb=HuberRegressor(alpha=1e-3,epsilon=1.35,max_iter=600).fit(gtr,y[tr])
                    oof['gram'][va]+=hb.predict(gva)/len(SEEDS); pred['gram']+=hb.predict(gte)/(N_FOLDS*len(SEEDS))
                except Exception:
                    rr=Ridge(alpha=1.0).fit(gtr,y[tr]); oof['gram'][va]+=rr.predict(gva)/len(SEEDS); pred['gram']+=rr.predict(gte)/(N_FOLDS*len(SEEDS))
    return members,oof,pred

def honest_eval(members, oof, pred, eval_idx):
    '''Outer-CV of the 2nd stage (NNLS+isotonic) on held-out rows; base OOF is single-pass (per-row
    honest). Returns fold-averaged metrics + genuinely held-out blend preds (oof) for the report.'''
    M=np.column_stack([oof[m][eval_idx] for m in members])
    keep=[i for i,m in enumerate(members) if np.isfinite(M[:,i]).all() and np.isfinite(pred[m]).all()]
    M=M[:,keep]; mem=[members[i] for i in keep]
    ye=y[eval_idx]; be=bins[eval_idx]; dure=dur_tr[eval_idx]; we=testlike_w(dure); short=dure<=DUR_SHORT_S
    Rr=[];Rt=[];Pr=[];Rs=[];Rl=[]
    ob_cv=np.zeros(len(ye)); ob_n=np.zeros(len(ye))   # genuinely held-out blend+isotonic preds (for the report)
    for so in OUTER_SEEDS:
        for tri,tei in StratifiedKFold(OUTER_FOLDS,shuffle=True,random_state=so).split(M,be):
            w,_=nnls(M[tri],ye[tri]); bt=np.clip(M[tri]@w,0,5); bv=np.clip(M[tei]@w,0,5)
            iso=IsotonicRegression(out_of_bounds='clip').fit(bt,ye[tri]); pv=np.clip(iso.predict(bv),0,5)
            ob_cv[tei]+=pv; ob_n[tei]+=1
            Rr.append(rmse(ye[tei],pv)); Rt.append(wrmse(ye[tei],pv,we[tei]))
            try: Pr.append(pearsonr(ye[tei],pv)[0])
            except Exception: pass
            if short[tei].sum()>3: Rs.append(rmse(ye[tei][short[tei]],pv[short[tei]]))
            if (~short[tei]).sum()>3: Rl.append(rmse(ye[tei][~short[tei]],pv[~short[tei]]))
    ob_cv/=np.maximum(ob_n,1)                          # average across the OUTER_SEEDS repeats -> held-out OOF
    # DEPLOYED model only: fit blend+calibration on ALL eval rows (use all data) -> test preds
    Mte_=np.column_stack([pred[m] for m in mem]); w,_=nnls(M,ye)
    iso=IsotonicRegression(out_of_bounds='clip').fit(np.clip(M@w,0,5),ye)
    test=np.clip(iso.predict(np.clip(Mte_@w,0,5)),0,5)
    return dict(rmse_rand=float(np.mean(Rr)),std_rand=float(np.std(Rr)),rmse_tl=float(np.mean(Rt)),
                pear=float(np.mean(Pr)) if Pr else float('nan'),
                rmse_short=float(np.mean(Rs)) if Rs else float('nan'),
                rmse_long=float(np.mean(Rl)) if Rl else float('nan'),
                rmse_pool=rmse(ye,ob_cv),pear_pool=float(pearsonr(ye,ob_cv)[0]),
                weights=dict(zip(mem,np.round(w,3))),test=test,oof=ob_cv,eval_idx=eval_idx)

idx_all=np.arange(len(y)); idx_pos=np.where(y>0)[0]
print('rows: all=%d  positive=%d  zeros=%d | test short-clip fraction=%.2f'%(len(idx_all),len(idx_pos),len(idx_all)-len(idx_pos),te_short_frac))
res={}
mA,oA,pA=train_stack(idx_all, handc_all, add_gram=True)
res['A all-769']        = honest_eval([m for m in mA if m!='gram'], oA,pA, idx_pos)
res['A all-769 +gram']  = honest_eval(mA, oA,pA, idx_pos)
if HANDLE_ZEROS:
    mB,oB,pB=train_stack(idx_pos, handc_all, add_gram=True)
    res['B pos-732']       = honest_eval([m for m in mB if m!='gram'], oB,pB, idx_pos)
    res['B pos-732 +gram'] = honest_eval(mB, oB,pB, idx_pos)
    if ABLATE_CTC:
        mBn,oBn,pBn=train_stack(idx_pos,[c for c in handc_all if not c.startswith(CTC_PREF)],add_gram=False)
        res['B pos-732 -CTC']= honest_eval(mBn,oBn,pBn, idx_pos)
print('\n(all scored on the 732 positive 1-5 targets; test-like = duration-reweighted to match test)')
print('%-20s %8s %8s %8s %8s %8s'%('config','tl-RMSE','randRMSE','Pearson','short','long'))
for k,r in res.items():
    print('%-20s %8.4f %8.4f %8.4f %8.4f %8.4f'%(k,r['rmse_tl'],r['rmse_rand'],r['pear'],r['rmse_short'],r['rmse_long']))
# ---- deploy: pre-registered default (drop zeros); de-biased vs winner's-curse min-selection ----
order=sorted(res,key=lambda k:res[k]['rmse_tl'])
default=DEPLOY_CONFIG if (DEPLOY_CONFIG in res) else order[0]
cand=order[0]; se=res[default]['std_rand']/np.sqrt(OUTER_FOLDS)
beats=res[cand]['rmse_tl'] < res[default]['rmse_tl']-SELECT_MARGIN_SE*se
best=cand if beats else default
print('\nlowest test-like RMSE: %s (%.4f) | default(pre-registered): %s (%.4f, ~fold-SE %.4f)'%(
      cand,res[cand]['rmse_tl'],default,res[default]['rmse_tl'],se))
if cand!=default:
    print('  delta(%s - %s)=%+.4f -> %s'%(cand,default,res[cand]['rmse_tl']-res[default]['rmse_tl'],
          'reliable win, deploying '+cand if beats else 'within noise, deploying default (confirm on LB)'))
print('>> DEPLOYED CONFIG: %s   weights: %s'%(best,res[best]['weights']))
preds=res[best]['test']; oof_blend=res[best]['oof']; eval_idx=res[best]['eval_idx']; y_eval=y[eval_idx]
BEST=res[best]
print('   HONESTY SCOPE: 2nd stage (NNLS blend + isotonic) held out on outer folds; base OOF single-pass')
print('   (per-row honest, trained once over the full set) -> compare configs by the DELTA above.')
print('   v5-vs-v6 is only comparable on the public LB (their OOF protocols differ).')

## 10 · Report & visualizations (interpretability is graded) — on the deployed config (held-out preds)

In [ ]:
import matplotlib.pyplot as plt
ob=oof_blend; ye=y_eval
fig,ax=plt.subplots(2,3,figsize=(17,9.5))
ax[0,0].scatter(ye,ob,s=14,alpha=0.5); ax[0,0].plot([1,5],[1,5],'r--',lw=1)
ax[0,0].set_xlabel('true'); ax[0,0].set_ylabel('held-out pred'); ax[0,0].set_title('Pred vs Actual (HELD-OUT)  RMSE=%.3f  r=%.3f | test-like=%.3f'%(BEST['rmse_pool'],BEST['pear_pool'],BEST['rmse_tl']))
dsub=dur_tr[eval_idx]
ax[0,1].scatter(dsub,ye-ob,s=12,alpha=0.5); ax[0,1].axhline(0,color='r',ls='--'); ax[0,1].axvline(DUR_SHORT_S,color='g',ls=':')
ax[0,1].set_xlabel('duration s'); ax[0,1].set_ylabel('residual (true-pred)'); ax[0,1].set_title('Residual vs duration (cohort check)')
ax[0,2].hist(ye,bins=20,alpha=0.6,label='true'); ax[0,2].hist(ob,bins=20,alpha=0.6,label='pred'); ax[0,2].legend(); ax[0,2].set_title('Score distribution (1-5)')
res_=ob-ye; ax[1,0].hist(res_,bins=25); ax[1,0].axvline(0,color='r',ls='--'); ax[1,0].set_title('Residuals (mean=%+.3f)'%res_.mean())
tb=np.clip(np.round(ye).astype(int),0,5); pb=np.clip(np.round(ob).astype(int),0,5); C=np.zeros((6,6))
for a,b in zip(tb,pb): C[a,b]+=1
ax[1,1].imshow(C,cmap='Blues'); ax[1,1].set_xlabel('pred band'); ax[1,1].set_ylabel('true band'); ax[1,1].set_title('Binned confusion')
for a in range(6):
    for b in range(6):
        if C[a,b]: ax[1,1].text(b,a,int(C[a,b]),ha='center',va='center',fontsize=8)
Aall=_v(Mtr,handc_all)
lgb_imp=lgb.train(lgb_params,lgb.Dataset(Aall[eval_idx],ye),num_boost_round=400)
full=pd.Series(lgb_imp.feature_importance(importance_type='gain'),index=handc_all)
imp=full.sort_values(ascending=False).head(18)[::-1]
ax[1,2].barh(range(len(imp)),imp.values); ax[1,2].set_yticks(range(len(imp))); ax[1,2].set_yticklabels(imp.index,fontsize=7); ax[1,2].set_title('Top handcrafted features (LGB gain)')
plt.tight_layout(); plt.savefig('report_v6.png',dpi=110); plt.show()

print('\n=== HONEST CONFIG COMPARISON (test-like RMSE; lower=better; compare by DELTA, confirm on LB) ===')
for k,r in res.items(): print('  %-20s tl-RMSE=%.4f (~SE %.4f)  rand=%.4f  Pearson=%.4f'%(k,r['rmse_tl'],r['std_rand']/np.sqrt(OUTER_FOLDS),r['rmse_rand'],r['pear']))
print('\n=== FEATURE-GROUP READOUT (share of LGB gain over handcrafted block) ===')
for name,pre in [('Whisper<->CTC divergence','dv_'),('CTC verbatim text','c_'),('GEC grammar','gec_'),
                 ('LLM judge','llm_'),('De-Jong fluency','dj_'),('ASR confidence','ac_'),('POS','pos_')]:
    cols=[c for c in handc_all if c.startswith(pre)]; s=float(full[cols].sum()) if cols else 0.0
    print('  %-26s: %5.1f%%'%(name,100*s/full.sum()))
print('\n=== WORST 10 HELD-OUT PREDICTIONS (positive targets; Whisper text) ===')
err=pd.DataFrame({FILE_COL:Mtr[FILE_COL].values[eval_idx],'true':ye,'pred':np.round(ob,2),'abs_err':np.round(np.abs(ob-ye),2)})
err=err.merge(tx_tr,on=FILE_COL).sort_values('abs_err',ascending=False).head(10)
for _,r in err.iterrows(): print('  %.2f | true %.1f pred %.2f | %s'%(r['abs_err'],r['true'],r['pred'],str(r['text'])[:88]))

## 11 · Final metrics + submission (honest training RMSE as the brief requires; keyed to test.csv, clipped 1-5)

In [ ]:
print('TRAINING RMSE (honest, TEST-LIKE, duration-standardised) = %.4f'%BEST['rmse_tl'])
print('TRAINING RMSE (honest, random-CV)   = %.4f   Pearson = %.4f'%(BEST['rmse_rand'],BEST['pear']))
print('held-out pooled RMSE = %.4f   pooled Pearson = %.4f'%(BEST['rmse_pool'],BEST['pear_pool']))
print('config deployed:',best,'| honesty: 2nd stage held out, base OOF single-pass; v5-vs-v6 only via LB')
sub=pd.DataFrame({FILE_COL:test[FILE_COL].values})
sub['label']=sub[FILE_COL].map(dict(zip(Mte[FILE_COL].values,preds))).fillna(float(np.mean(y_eval)))
sub['label']=np.clip(sub['label'],1,5)   # rubric is 1-5; verified test has no zeros
sub.to_csv('submission.csv',index=False)
assert len(sub)==len(test) and sub['label'].notna().all(), 'submission must have one row per test file'
print(sub.head()); print('rows:',len(sub),'range:',round(sub.label.min(),3),'..',round(sub.label.max(),3))